In [8]:
import os
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel
from langchain_core.prompts import ChatPromptTemplate
load_dotenv()
api_key = os.getenv("API_KEY")

In [2]:
chat = ChatOllama(
    model="gpt-oss:120b",
    temperature=0.7,
    base_url="https://ollama.com",
    client_kwargs={
        "headers": {
            "Authorization": f"Bearer {api_key}"
        }
    }
)

In [3]:
chat_template_books = ChatPromptTemplate.from_template(
    '''
    Suggest three of the best intermediate-level {programming language} books. 
    Answer only by listing the books.
    '''
)

chat_template_projects = ChatPromptTemplate.from_template(
    '''
    Suggest three interesting {programming language} projects suitable for intermediate-level programmers. 
    Answer only by listing the projects.
    '''
)

In [4]:
string_parser = StrOutputParser()


In [5]:
chain_books = chat_template_books | chat | string_parser


In [6]:
chain_projects = chat_template_projects | chat | string_parser


In [9]:
chain_parallel = RunnableParallel({'books':chain_books, 'projects':chain_projects})


In [10]:
chain_parallel.invoke({'programming language':'Python'})

{'books': '- Fluent Python  \n- Effective Python  \n- Python Cookbook',
 'projects': '1. Personal finance tracker with budgeting and visualization  \n2. Web‑based collaborative markdown editor with real‑time preview  \n3. AI‑powered chatbot that integrates with popular messaging platforms  '}

## Another example

In [11]:
chat_template_books = ChatPromptTemplate.from_template(
    '''
    Suggest three of the best intermediate-level {programming language} books. 
    Answer only by listing the books.
    '''
)

chat_template_projects = ChatPromptTemplate.from_template(
    '''
    Suggest three interesting {programming language} projects suitable for intermediate-level programmers. 
    Answer only by listing the projects.
    '''
)

chat_template_time = ChatPromptTemplate.from_template(
     '''
     I'm an intermediate level programmer.
     
     Consider the following literature:
     {books}
     
     Also, consider the following projects:
     {projects}
     
     Roughly how much time would it take me to complete the literature and the projects?
     
     '''
)

In [17]:
chain_books = chat_template_books | chat | string_parser

chain_projects = chat_template_projects | chat | string_parser

# chain_parallel = RunnableParallel({'books':chain_books, 'projects':chain_projects})

In [ ]:
# chain_parallel.invoke({'programming language':'Python'})

{'books': 'Fluent Python  \nEffective Python  \nPython Cookbook',
 'projects': '1. Personal finance tracker with data visualization  \n2. Web‑based markdown editor with live preview  \n3. AI‑powered chatbot that learns from user conversations'}

In [18]:
chain_time1 = (RunnableParallel({'books':chain_books, 
                                'projects':chain_projects}) 
              | chat_template_time 
              | chat 
              | string_parser
             )

In [19]:
print(chain_time1.invoke({'programming language':'Python'}))

Below is a **back‑of‑the‑envelope schedule** that breaks the three books and three projects into bite‑size chunks, adds a safety margin for learning‑by‑doing, and shows how the total effort translates into weeks or months for a typical “intermediate” Python programmer (someone who is comfortable with the language fundamentals, can write clean scripts, and has done a few small‑to‑medium projects on their own).

---

## 1.  The literature

| Book | What you’ll actually do | Estimated time* |
|------|------------------------|-----------------|
| **Fluent Python** (≈ 800 pp) | • Skim each chapter → identify the “core” concepts (data model, iterators, descriptors, coroutines, etc.)  <br>• Do the “What‑to‑do‑next” exercises (most chapters have 2‑4)  <br>• Write a tiny demo (5‑10 min) for each new idiom | **30 – 45 h** |
| **Effective Python** (≈ 300 pp, 90 items) | • Read 3‑4 items per day (≈ 5 min each)  <br>• Rewrite a piece of your own code to use the pattern (≈ 10 min)  <br>• Keep a “che

In [16]:
chain_time1.get_graph().print_ascii()

            +-------------------------------+              
            | Parallel<books,projects>Input |              
            +-------------------------------+              
                   ***               ***                   
                ***                     ***                
              **                           **              
+--------------------+              +--------------------+ 
| ChatPromptTemplate |              | ChatPromptTemplate | 
+--------------------+              +--------------------+ 
           *                                   *           
           *                                   *           
           *                                   *           
    +------------+                      +------------+     
    | ChatOllama |                      | ChatOllama |     
    +------------+                      +------------+     
           *                                   *           
           *                            